In [1]:
import pdfplumber
import re
from tqdm.notebook import tqdm
import pandas as pd
import os

In [2]:
# read pdf and csv files
pdffiles = [file for file in os.listdir('./data/pdf') if file.endswith('.pdf')]
csvfiles = [file for file in os.listdir('./data/raw') if file.endswith('.csv')]
print(pdffiles)

['im4_2025_12.pdf', 'im4_2026_01.pdf', 'im4_2026_02.pdf', 'im4_2026_03.pdf', 'im4_2026_04.pdf', 'im7_2025_12.pdf', 'im7_2026_01.pdf', 'im7_2026_02.pdf', 'im7_2026_03.pdf', 'im7_2026_04.pdf']


In [3]:
# check if csv already exists for the month
files_to_extract = []
for file in pdffiles:
    csvfile = file.replace('.pdf', '.csv')
    if csvfile not in csvfiles:
        files_to_extract.append(file)
print(files_to_extract)

['im4_2026_04.pdf', 'im7_2026_04.pdf']


In [4]:
# extract csv for pending months
file_store = {}

for file in files_to_extract:
    print(file)
    data_type = file[0:3]
    data_year = file[4:8]
    data_month = file[9:11]
    pdf = pdfplumber.open(f'data/pdf/{file}')
    pages = pdf.pages
    print(len(pages))
    store = []
    for page in tqdm(pages):
        tables = page.extract_tables()
        if tables:
            for table in tables:
                store.append(table)
    print(len(store))
    file_store[file] = store

im4_2026_04.pdf
66


  0%|          | 0/66 [00:00<?, ?it/s]

66
im7_2026_04.pdf
31


  0%|          | 0/31 [00:00<?, ?it/s]

31


In [5]:
# print table header of each page to check consistency of table
for file in file_store:
    store = file_store[file]
    for table in store:
        print(table[0])

['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']

In [6]:
# combine tables of each page into single dataframe for each pdf
file_df = {}
for file in file_store:
    store = file_store[file]
    rows = []
    columns = store[0][0]
    for table in store:
        for row in table[1:]:
            rows.append(row)
    print(len(rows))
    df = pd.DataFrame(rows, columns=columns)
    file_df[file] = df

4250
2119


In [7]:
# rename dataframe columns
file_cols = ['HSCODE', 'Tarrif Description', 'Net_WT (kg)', 'Assess Value (Tk)', 'Invoice Value (Tk)']
csv_cols = ['hscode', 'description', 'net_wt_kg', 'assess_value_bdt', 'invoice_value_bdt']

for file in file_df:
    df = file_df[file]
    rename_cols = {}
    for i, col in enumerate(file_cols):
        if col in df.columns:
            rename_cols[col] = csv_cols[i]
    df.rename(columns=rename_cols, inplace=True)

In [8]:
# function to check values in dataframe
def check(row):
    if not re.search(r'\d{8}', row.hscode):
        print(row.hscode)
    if not re.search(r'\d+', row.net_wt_kg.replace(',', '')):
        print(row.net_wt_kg)
    if not re.search(r'\d+', row.assess_value_bdt.replace(',', '')):
        print(row.assess_value_bdt)
    if 'invoice_value_bdt' in row:
        if not re.search(r'\d+', row.invoice_value_bdt.replace(',', '')):
            print(row.invoice_value_bdt)

In [9]:
# check values in dataframe
for file in file_df:
    print(file)
    df = file_df[file]
    a = df.apply(check, axis=1)

im4_2026_04.pdf
Grand Total
im7_2026_04.pdf
Grand Total


In [10]:
# drop inconsistent rows from dataframe
for file in file_df:
    print(file)
    df = file_df[file]
    print(df.shape)
    df.drop(df[df.hscode=='Grand Total'].index, inplace=True)
    file_df[file] = df
    print(df.shape)

im4_2026_04.pdf
(4250, 5)
(4249, 5)
im7_2026_04.pdf
(2119, 5)
(2118, 5)


In [11]:
# replace comma from numeric values
for file in file_df:
    print(file)
    df = file_df[file]
    df['net_wt_kg'] = df['net_wt_kg'].str.replace(',', '')
    df['assess_value_bdt'] = df['assess_value_bdt'].str.replace(',', '')
    if 'invoice_value_bdt' in df.columns:
        df['invoice_value_bdt'] = df['invoice_value_bdt'].str.replace(',', '')
    file_df[file] = df

im4_2026_04.pdf
im7_2026_04.pdf


In [12]:
for file in file_df:
    print(file)
    df = file_df[file]
    df.to_csv(f'data/raw/{file.replace(".pdf", "")}.csv', index=False)

im4_2026_04.pdf
im7_2026_04.pdf
